# Forward Contract Analysis

This notebook evaluates the impact of forward contract hedging for a multi-currency FX exposure.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

data_path = Path.cwd().parent / 'Casestudy_Data.xlsx'
data = pd.read_excel(data_path, sheet_name='Data', parse_dates=['Date'])
data.head()

,Date,CCYPair,Expiry,Spot,Forward,ATM Vol
0,2024-07-01,USDCNH,1Y,7.30555,7.119950,0.058250
1,2024-07-01,USDINR,1Y,83.47000,84.837663,0.034972
2,2024-07-01,USDJPY,1Y,161.61500,153.565416,0.099414
3,2024-07-01,USDKRW,1Y,1384.48000,1356.255408,0.087683
4,2024-07-02,USDCNH,1Y,7.30690,7.123650,0.058966


In [2]:
latest_data = data.sort_values('Date').groupby('CCYPair', as_index=False).tail(1)
latest_data[['Date', 'CCYPair', 'Spot', 'Forward']].to_string(index=False)

'      Date CCYPair      Spot     Forward\n2026-06-30  USDINR   94.6396   97.444607\n2026-06-30  USDCNH    6.7892    6.608950\n2026-06-30  USDJPY  162.5100  157.696811\n2026-06-30  USDKRW 1548.8753 1536.744475'

In [3]:
investment_usd = 250_000_000
summary_rows = []

for _, row in latest_data.iterrows():
    pair = row['CCYPair']
    spot = float(row['Spot'])
    forward = float(row['Forward'])
    units = investment_usd / spot
    summary_rows.append({
        'CCYPair': pair,
        'Spot': spot,
        'Forward': forward,
        'Investment_USD': investment_usd,
        'Foreign_Units': units,
        'Forward_Hedge_USD': units * forward,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

,CCYPair,Spot,Forward,Investment_USD,Foreign_Units,Forward_Hedge_USD
0,USDINR,94.6396,97.444607,250000000,2.641600e+06,2.574097e+08
1,USDCNH,6.7892,6.608950,250000000,3.682319e+07,2.433626e+08
2,USDJPY,162.5100,157.696811,250000000,1.538367e+06,2.425955e+08
3,USDKRW,1548.8753,1536.744475,250000000,1.614074e+05,2.480420e+08


In [4]:
scenarios = [-30, -20, -15, -10, -5, 0, 5, 10]
scenario_rows = []

for _, row in summary_df.iterrows():
    for scenario_pct in scenarios:
        future_spot = row['Spot'] * (1 + scenario_pct / 100)
        no_hedge_usd = row['Foreign_Units'] * future_spot
        forward_hedge_usd = row['Foreign_Units'] * row['Forward']
        scenario_rows.append({
            'CCYPair': row['CCYPair'],
            'FX_Scenario': f'{scenario_pct:+.0f}%',
            'Future_Spot': future_spot,
            'No_Hedge_USD': no_hedge_usd,
            'Forward_Hedge_USD': forward_hedge_usd,
            'Forward_Contract_Gain_USD': forward_hedge_usd - no_hedge_usd,
        })

scenario_df = pd.DataFrame(scenario_rows)
scenario_df.head()

,CCYPair,FX_Scenario,Future_Spot,No_Hedge_USD,Forward_Hedge_USD,Forward_Contract_Gain_USD
0,USDINR,-30%,66.24772,175000000.0,2.574097e+08,8.240971e+07
1,USDINR,-20%,75.71168,200000000.0,2.574097e+08,5.740971e+07
2,USDINR,-15%,80.44366,212500000.0,2.574097e+08,4.490971e+07
3,USDINR,-10%,85.17564,225000000.0,2.574097e+08,3.240971e+07
4,USDINR,-5%,89.90762,237500000.0,2.574097e+08,1.990971e+07


In [5]:
output_path = Path.cwd() / 'forward_contract_analysis_results.xlsx'
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    summary_df.to_excel(writer, sheet_name='Forward_Summary', index=False)
    scenario_df.to_excel(writer, sheet_name='Forward_Scenarios', index=False)

print(f'Results saved to: {output_path}')

Results saved to: c:\Users\shahk\OneDrive\Desktop\BIIC\Final_Submission\ForwardContractAnalysis\forward_contract_analysis_results.xlsx
